In [17]:
import pandas as pd
import plotly.graph_objects as go
import re

# Sample airport coordinates dictionary (expand as needed)
airport_coords = {
    'LHR': (51.4700, -0.4543),   # London Heathrow
    'DXB': (25.2532, 55.3657),   # Dubai International
    'SIN': (1.3644, 103.9915),   # Singapore Changi
    'FRA': (50.0379, 8.5622),    # Frankfurt
    'BOM': (19.0896, 72.8656),   # Mumbai
    'CDG': (49.0097, 2.5479),    # Paris Charles de Gaulle
    'DOH': (25.2731, 51.6085),   # Doha Hamad
    'AUH': (24.4539, 54.3776),   # Abu Dhabi
    'IST': (41.2753, 28.7519),   # Istanbul
    'SVO': (55.9726, 37.4146),   # Moscow Sheremetyevo
    'DEL': (28.5562, 77.1000),   # Delhi
    'KHI': (24.9065, 67.1604),   # Karachi
    'MCT': (23.5937, 58.2844),   # Muscat
    'AMS': (52.3105, 4.7683),    # Amsterdam
    'CAI': (30.1219, 31.4056),   # Cairo
    'MAD': (40.4720, -3.5706),   # Madrid
    'HKG': (22.3080, 113.9185),  # Hong Kong
    'ICN': (37.4602, 126.4407),  # Seoul Incheon
    'KUL': (2.7456, 101.7090),   # Kuala Lumpur
    'BAH': (26.2708, 50.6336),   # Bahrain
    'RUH': (24.9576, 46.6986),   # Riyadh
    'BEY': (33.8209, 35.4884),   # Beirut
    'BKK': (13.6900, 100.7501),  # Bangkok
}

# Load reroute data
flight_rerouted = pd.read_csv('Aviation_disruption/flight_reroutes.csv')

# Function to extract airport codes from route string
def extract_code(route):
    match = re.findall(r'\b[A-Z]{3}\b', route)
    return match[0] if match else None

def extract_code_dest(route):
    match = re.findall(r'\b[A-Z]{3}\b', route)
    return match[-1] if match else None

# Extract codes for origin and destination
flight_rerouted['orig_code'] = flight_rerouted['original_route'].apply(extract_code)
flight_rerouted['dest_code'] = flight_rerouted['original_route'].apply(extract_code_dest)
flight_rerouted['div_orig_code'] = flight_rerouted['diverted_route'].apply(extract_code)
flight_rerouted['div_dest_code'] = flight_rerouted['diverted_route'].apply(extract_code_dest)

# Map codes to coordinates
flight_rerouted['original_lat'] = flight_rerouted['orig_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['original_lon'] = flight_rerouted['orig_code'].map(lambda x: airport_coords.get(x, (None, None))[1])
flight_rerouted['diverted_lat'] = flight_rerouted['div_orig_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['diverted_lon'] = flight_rerouted['div_orig_code'].map(lambda x: airport_coords.get(x, (None, None))[1])

# Drop rows with missing coordinates
flight_rerouted = flight_rerouted.dropna(subset=['original_lat', 'original_lon', 'diverted_lat', 'diverted_lon'])

# Plot reroutes as lines on map
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row['original_lon'], row['diverted_lon']],
        lat=[row['original_lat'], row['diverted_lat']],
        mode='lines+markers',
        line=dict(width=2, color='blue'),
        marker=dict(size=8),
        name=row['airline'],
        hoverinfo='text',
        text=f"Flight: {row['flight_id']}<br>Delay: {row['delay_minutes']} min"
    ))

fig.update_layout(
    title='Flight Reroutes Due to Conflict',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()

In [18]:
# Plot reroutes as lines and destination points on map
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    # Line from original to diverted destination
    fig.add_trace(go.Scattergeo(
        lon=[row['original_lon'], row['diverted_lon']],
        lat=[row['original_lat'], row['diverted_lat']],
        mode='lines',
        line=dict(width=2, color='blue'),
        showlegend=False,
        hoverinfo='text',
        text=f"Flight: {row['flight_id']}<br>Delay: {row['delay_minutes']} min"
    ))
    # Original destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['original_lon']],
        lat=[row['original_lat']],
        mode='markers',
        marker=dict(size=10, color='green', symbol='circle'),
        name='Original Destination',
        showlegend=(idx==0)
    ))
    # Diverted destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['diverted_lon']],
        lat=[row['diverted_lat']],
        mode='markers',
        marker=dict(size=10, color='red', symbol='x'),
        name='Diverted Destination',
        showlegend=(idx==0)
    ))

fig.update_layout(
    title='Flight Reroutes: Original vs Diverted Destinations',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()

In [19]:
# Add coordinates for waypoints
waypoint_coords = {
    'Gulf': (25.0, 50.0),           # Example coordinates for Gulf
    'Southern Red Sea': (15.0, 42.0) # Example coordinates for Southern Red Sea
}

def extract_waypoint(route):
    if 'Gulf' in route:
        return 'Gulf'
    elif 'Red Sea' in route:
        return 'Southern Red Sea'
    else:
        return None

flight_rerouted['orig_dest_code'] = flight_rerouted['original_route'].apply(lambda r: re.findall(r'\b[A-Z]{3}\b', r)[1])
flight_rerouted['div_dest_code'] = flight_rerouted['diverted_route'].apply(lambda r: re.findall(r'\b[A-Z]{3}\b', r)[1])
flight_rerouted['orig_waypoint'] = flight_rerouted['original_route'].apply(extract_waypoint)
flight_rerouted['div_waypoint'] = flight_rerouted['diverted_route'].apply(extract_waypoint)

# Map to coordinates
flight_rerouted['original_lat'] = flight_rerouted['orig_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['original_lon'] = flight_rerouted['orig_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[1])
flight_rerouted['diverted_lat'] = flight_rerouted['div_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['diverted_lon'] = flight_rerouted['div_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[1])
flight_rerouted['orig_way_lat'] = flight_rerouted['orig_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[0])
flight_rerouted['orig_way_lon'] = flight_rerouted['orig_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[1])
flight_rerouted['div_way_lat'] = flight_rerouted['div_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[0])
flight_rerouted['div_way_lon'] = flight_rerouted['div_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[1])

# Plot with waypoints
fig = go.Figure()
for idx, row in flight_rerouted.iterrows():
    # Original route (with waypoint)
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['original_lon'], row['orig_way_lon']],
            lat=[row['original_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=False,
            hoverinfo='text',
            text='Original Route via ' + str(row['orig_waypoint'])
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_way_lon'], row['original_lon']],
            lat=[row['orig_way_lat'], row['original_lat']],
            mode='markers',
            marker=dict(size=10, color='green', symbol='circle'),
            name='Original Destination',
            showlegend=(idx==0)
        ))
    # Diverted route (with waypoint)
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['diverted_lon'], row['div_way_lon']],
            lat=[row['diverted_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=False,
            hoverinfo='text',
            text='Diverted Route via ' + str(row['div_waypoint'])
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon'], row['diverted_lon']],
            lat=[row['div_way_lat'], row['diverted_lat']],
            mode='markers',
            marker=dict(size=10, color='red', symbol='x'),
            name='Diverted Destination',
            showlegend=(idx==0)
        ))

fig.update_layout(
    title='Flight Reroutes with Route Waypoints',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()


In [20]:
import pandas as pd
import plotly.graph_objects as go
import re

# Airport coordinates dictionary (expand as needed)
airport_coords = {
    'LHR': (51.4700, -0.4543),   # London Heathrow
    'DXB': (25.2532, 55.3657),   # Dubai International
    'SIN': (1.3644, 103.9915),   # Singapore Changi
    'FRA': (50.0379, 8.5622),    # Frankfurt
    'BOM': (19.0896, 72.8656),   # Mumbai
    'CDG': (49.0097, 2.5479),    # Paris Charles de Gaulle
    'DOH': (25.2731, 51.6085),   # Doha Hamad
    'AUH': (24.4539, 54.3776),   # Abu Dhabi
    'IST': (41.2753, 28.7519),   # Istanbul
    'SVO': (55.9726, 37.4146),   # Moscow Sheremetyevo
    'DEL': (28.5562, 77.1000),   # Delhi
    'KHI': (24.9065, 67.1604),   # Karachi
    'MCT': (23.5937, 58.2844),   # Muscat
    'AMS': (52.3105, 4.7683),    # Amsterdam
    'CAI': (30.1219, 31.4056),   # Cairo
    'MAD': (40.4720, -3.5706),   # Madrid
    'HKG': (22.3080, 113.9185),  # Hong Kong
    'ICN': (37.4602, 126.4407),  # Seoul Incheon
    'KUL': (2.7456, 101.7090),   # Kuala Lumpur
    'BAH': (26.2708, 50.6336),   # Bahrain
    'RUH': (24.9576, 46.6986),   # Riyadh
    'BEY': (33.8209, 35.4884),   # Beirut
    'BKK': (13.6900, 100.7501),  # Bangkok
}

# Example coordinates for waypoints (expand as needed)
waypoint_coords = {
    'Gulf': (25.0, 50.0),
    'Red Sea': (15.0, 42.0),
    'Southern Red Sea': (13.0, 43.0),
    'Caspian': (41.7, 52.0),
    'Black Sea': (43.0, 35.0),
    'Egypt': (26.0, 30.0),
    'Africa': (10.0, 20.0),
    'India Ocean': (0.0, 80.0),
    'Colombo': (6.9271, 79.8612),
    'Muscat': (23.6100, 58.5400),
    'Oman': (20.0, 57.0),
    'Caucasus': (42.0, 45.0),
    'Turkey': (39.0, 35.0),
    'Siberia': (60.0, 105.0),
    'Northern Europe': (55.0, 15.0),
    'Southern Indian Ocean': (-20.0, 90.0),
    'Red Sea coastal': (20.0, 38.0),
    'Egyptian route': (27.0, 31.0),
    'Saudi': (24.0, 45.0),
    'Africa/Southern route': (0.0, 25.0),
    'Oman/India': (15.0, 75.0),
    'southern deviation': (10.0, 40.0),
    'standard Gulf': (25.0, 50.0),
    'standard': (25.0, 50.0),
}

# Load reroute data
flight_rerouted = pd.read_csv('Aviation_disruption/flight_reroutes.csv')

def extract_airport_codes(route):
    codes = re.findall(r'\b[A-Z]{3}\b', route)
    return codes[0], codes[1] if len(codes) > 1 else (None, None)

def extract_waypoint(route):
    # Find the "via ..." part
    via_match = re.search(r'via ([^,/]*)', route)
    if via_match:
        via = via_match.group(1).strip()
        # Try to match to known waypoints
        for key in waypoint_coords.keys():
            if key.lower() in via.lower():
                return key
        return via  # fallback
    return None

# Extract origin, destination, and waypoints
flight_rerouted['orig_code'], flight_rerouted['dest_code'] = zip(*flight_rerouted['original_route'].apply(extract_airport_codes))
flight_rerouted['div_orig_code'], flight_rerouted['div_dest_code'] = zip(*flight_rerouted['diverted_route'].apply(extract_airport_codes))
flight_rerouted['orig_waypoint'] = flight_rerouted['original_route'].apply(extract_waypoint)
flight_rerouted['div_waypoint'] = flight_rerouted['diverted_route'].apply(extract_waypoint)

# Map to coordinates
flight_rerouted['orig_lat'] = flight_rerouted['dest_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['orig_lon'] = flight_rerouted['dest_code'].map(lambda x: airport_coords.get(x, (None, None))[1])
flight_rerouted['div_lat'] = flight_rerouted['div_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[0])
flight_rerouted['div_lon'] = flight_rerouted['div_dest_code'].map(lambda x: airport_coords.get(x, (None, None))[1])
flight_rerouted['orig_way_lat'] = flight_rerouted['orig_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[0])
flight_rerouted['orig_way_lon'] = flight_rerouted['orig_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[1])
flight_rerouted['div_way_lat'] = flight_rerouted['div_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[0])
flight_rerouted['div_way_lon'] = flight_rerouted['div_waypoint'].map(lambda x: waypoint_coords.get(x, (None, None))[1])

# Drop rows with missing coordinates
flight_rerouted = flight_rerouted.dropna(subset=['orig_lat', 'orig_lon', 'div_lat', 'div_lon'])

# Plot with waypoints
fig = go.Figure()
for idx, row in flight_rerouted.iterrows():
    # Original route (with waypoint)
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_lon'], row['orig_way_lon']],
            lat=[row['orig_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=False,
            hoverinfo='text',
            text='Original Route via ' + str(row['orig_waypoint'])
        ))
    # Diverted route (with waypoint)
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['div_lon'], row['div_way_lon']],
            lat=[row['div_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=False,
            hoverinfo='text',
            text='Diverted Route via ' + str(row['div_waypoint'])
        ))
    # Mark original destination
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers',
        marker=dict(size=10, color='green', symbol='circle'),
        name='Original Destination',
        showlegend=(idx==0)
    ))
    # Mark diverted destination
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers',
        marker=dict(size=10, color='red', symbol='x'),
        name='Diverted Destination',
        showlegend=(idx==0)
    ))

fig.update_layout(
    title='Flight Reroutes with Route Waypoints',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()

In [21]:
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    # Original route: destination via original waypoint
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_lon'], row['orig_way_lon']],
            lat=[row['orig_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=False,
            hoverinfo='text',
            text=f"Original Route via {row['orig_waypoint']}"
        ))
        # Waypoint marker
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_way_lon']],
            lat=[row['orig_way_lat']],
            mode='markers',
            marker=dict(size=12, color='blue', symbol='diamond'),
            name='Original Waypoint',
            showlegend=(idx==0)
        ))
    # Diverted route: destination via diverted waypoint
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['div_lon'], row['div_way_lon']],
            lat=[row['div_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=False,
            hoverinfo='text',
            text=f"Diverted Route via {row['div_waypoint']}"
        ))
        # Waypoint marker
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon']],
            lat=[row['div_way_lat']],
            mode='markers',
            marker=dict(size=12, color='blue', symbol='diamond'),
            name='Diverted Waypoint',
            showlegend=(idx==0)
        ))
    # Original destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers',
        marker=dict(size=14, color='green', symbol='circle'),
        name='Original Destination',
        showlegend=(idx==0)
    ))
    # Diverted destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers',
        marker=dict(size=14, color='red', symbol='x'),
        name='Diverted Destination',
        showlegend=(idx==0)
    ))

fig.update_layout(
    title='Flight Reroutes: Clear Route and Waypoint Visualization',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()

In [22]:
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    # Draw reroute line (original destination to diverted destination)
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon'], row['div_lon']],
        lat=[row['orig_lat'], row['div_lat']],
        mode='lines',
        line=dict(width=3, color='blue'),
        showlegend=False,
        hoverinfo='text',
        text=f"Flight: {row['flight_id']}<br>Delay: {row['delay_minutes']} min"
    ))
    # Original destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers',
        marker=dict(size=14, color='green', symbol='circle'),
        name='Original Destination',
        showlegend=(idx==0)
    ))
    # Diverted destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers',
        marker=dict(size=14, color='red', symbol='x'),
        name='Diverted Destination',
        showlegend=(idx==0)
    ))
    # Waypoint marker (if present)
    if row.get('div_way_lat') and row.get('div_way_lon'):
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon']],
            lat=[row['div_way_lat']],
            mode='markers',
            marker=dict(size=12, color='blue', symbol='diamond'),
            name='Diverted Waypoint',
            showlegend=(idx==0)
        ))

fig.update_layout(
    title='Flight Reroutes: Original vs Diverted Destination',
    geo=dict(scope='world'),
    width=1200,
    height=700
)
fig.show()

In [23]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# Unique color per airline
airlines = flight_rerouted['airline'].unique()
color_map = plt.cm.get_cmap('tab20', len(airlines))
airline_colors = {airline: f"rgb{tuple([int(x*255) for x in color_map(i)[:3]])}" for i, airline in enumerate(airlines)}

fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Original route line and waypoint (red)
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_lon'], row['orig_way_lon']],
            lat=[row['orig_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=(idx==0),
            name='Original Route',
            hoverinfo='text',
            text=f"Original Route via {row['orig_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_way_lon']],
            lat=[row['orig_way_lat']],
            mode='markers',
            marker=dict(size=12, color='red', symbol='diamond'),
            name='Original Waypoint',
            showlegend=(idx==0)
        ))
    # Diverted route line and waypoint (green)
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['div_lon'], row['div_way_lon']],
            lat=[row['div_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=(idx==0),
            name='Diverted Route',
            hoverinfo='text',
            text=f"Diverted Route via {row['div_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon']],
            lat=[row['div_way_lat']],
            mode='markers',
            marker=dict(size=12, color='green', symbol='diamond'),
            name='Diverted Waypoint',
            showlegend=(idx==0)
        ))
    # Original destination marker (circle, airline color)
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers',
        marker=dict(size=14, color=color, symbol='circle'),
        name=f'Original Destination ({airline})',
        showlegend=(idx==0)
    ))
    # Diverted destination marker (x, airline color)
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers',
        marker=dict(size=14, color=color, symbol='x'),
        name=f'Diverted Destination ({airline})',
        showlegend=(idx==0)
    ))

fig.update_layout(
    title='Flight Reroutes: Airline-Colored Destinations, Red/Green Routes',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()

C:\Users\ramya\AppData\Local\Temp\ipykernel_11244\2450928271.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = plt.cm.get_cmap('tab20', len(airlines))


In [24]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# Unique color per airline
airlines = flight_rerouted['airline'].unique()
color_map = plt.cm.get_cmap('tab20', len(airlines))
airline_colors = {airline: f"rgb{tuple([int(x*255) for x in color_map(i)[:3]])}" for i, airline in enumerate(airlines)}

fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Original route line and waypoint (red)
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_lon'], row['orig_way_lon']],
            lat=[row['orig_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=(idx==0),
            name='Original Route',
            hoverinfo='text',
            text=f"Original Route via {row['orig_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_way_lon']],
            lat=[row['orig_way_lat']],
            mode='markers',
            marker=dict(size=12, color='red', symbol='diamond'),
            name='Original Waypoint',
            showlegend=(idx==0)
        ))
    # Diverted route line and waypoint (green)
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['div_lon'], row['div_way_lon']],
            lat=[row['div_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=(idx==0),
            name='Diverted Route',
            hoverinfo='text',
            text=f"Diverted Route via {row['div_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon']],
            lat=[row['div_way_lat']],
            mode='markers',
            marker=dict(size=12, color='green', symbol='diamond'),
            name='Diverted Waypoint',
            showlegend=(idx==0)
        ))
    # Original destination marker (flight symbol, airline color)
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers',
        marker=dict(size=18, color=color, symbol='airport'),
        name=f'Original Destination ({airline})',
        showlegend=(idx==0)
    ))
    # Diverted destination marker (flight symbol, airline color)
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers',
        marker=dict(size=18, color=color, symbol='airport'),
        name=f'Diverted Destination ({airline})',
        showlegend=(idx==0)
    ))

fig.update_layout(
    title='Flight Reroutes: Airline-Colored Flight Destinations, Red/Green Waypoints',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()

C:\Users\ramya\AppData\Local\Temp\ipykernel_11244\1393666701.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = plt.cm.get_cmap('tab20', len(airlines))


ValueError: 
    Invalid value of type 'builtins.str' received for the 'symbol' property of scattergeo.marker
        Received value: 'airport'

    The 'symbol' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            [0, '0', 'circle', 100, '100', 'circle-open', 200, '200',
            'circle-dot', 300, '300', 'circle-open-dot', 1, '1',
            'square', 101, '101', 'square-open', 201, '201',
            'square-dot', 301, '301', 'square-open-dot', 2, '2',
            'diamond', 102, '102', 'diamond-open', 202, '202',
            'diamond-dot', 302, '302', 'diamond-open-dot', 3, '3',
            'cross', 103, '103', 'cross-open', 203, '203',
            'cross-dot', 303, '303', 'cross-open-dot', 4, '4', 'x',
            104, '104', 'x-open', 204, '204', 'x-dot', 304, '304',
            'x-open-dot', 5, '5', 'triangle-up', 105, '105',
            'triangle-up-open', 205, '205', 'triangle-up-dot', 305,
            '305', 'tr

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# Unique color per airline
airlines = flight_rerouted['airline'].unique()
color_map = plt.cm.get_cmap('tab20', len(airlines))
airline_colors = {airline: f"rgb{tuple([int(x*255) for x in color_map(i)[:3]])}" for i, airline in enumerate(airlines)}

fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Original route line and waypoint (red)
    if row['orig_way_lat'] and row['orig_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_lon'], row['orig_way_lon']],
            lat=[row['orig_lat'], row['orig_way_lat']],
            mode='lines',
            line=dict(width=2, color='red'),
            showlegend=(idx==0),
            name='Original Route',
            hoverinfo='text',
            text=f"Original Route via {row['orig_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['orig_way_lon']],
            lat=[row['orig_way_lat']],
            mode='markers',
            marker=dict(size=12, color='red', symbol='diamond'),
            name='Original Waypoint',
            showlegend=(idx==0)
        ))
    # Diverted route line and waypoint (green)
    if row['div_way_lat'] and row['div_way_lon']:
        fig.add_trace(go.Scattergeo(
            lon=[row['div_lon'], row['div_way_lon']],
            lat=[row['div_lat'], row['div_way_lat']],
            mode='lines',
            line=dict(width=2, color='green'),
            showlegend=(idx==0),
            name='Diverted Route',
            hoverinfo='text',
            text=f"Diverted Route via {row['div_waypoint']}"
        ))
        fig.add_trace(go.Scattergeo(
            lon=[row['div_way_lon']],
            lat=[row['div_way_lat']],
            mode='markers',
            marker=dict(size=12, color='green', symbol='diamond'),
            name='Diverted Waypoint',
            showlegend=(idx==0)
        ))
    # # Original destination marker (flight symbol, airline color)
    # fig.add_trace(go.Scattergeo(
    #     lon=[row['orig_lon']],
    #     lat=[row['orig_lat']],
    #     mode='markers',
    #     marker=dict(size=18, color=color, symbol='airport'),
    #     name=f'Original Destination ({airline})',
    #     showlegend=(idx==0)
    # ))
    # # Diverted destination marker (flight symbol, airline color)
    # fig.add_trace(go.Scattergeo(
    #     lon=[row['div_lon']],
    #     lat=[row['div_lat']],
    #     mode='markers',
    #     marker=dict(size=18, color=color, symbol='airport'),
    #     name=f'Diverted Destination ({airline})',
    #     showlegend=(idx==0)
    # ))

    fig.add_trace(go.Scattergeo(
    lon=[row['orig_lon']],
    lat=[row['orig_lat']],
    mode='markers',
    marker=dict(size=18, color=color, symbol='star'),  # or 'circle'
    name=f'Original Destination ({airline})',
    showlegend=(idx==0)
))
fig.add_trace(go.Scattergeo(
    lon=[row['div_lon']],
    lat=[row['div_lat']],
    mode='markers',
    marker=dict(size=18, color=color, symbol='star'),  # or 'circle'
    name=f'Diverted Destination ({airline})',
    showlegend=(idx==0)
))

fig.update_layout(
    title='Flight Reroutes: Airline-Colored Flight Destinations, Red/Green Waypoints',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()

In [ ]:
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Draw line from origin to original destination
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon'], row['div_lon']],
        lat=[row['orig_lat'], row['div_lat']],
        mode='lines',
        line=dict(width=2, color=color),
        showlegend=False,
        hoverinfo='text',
        text=f"{row['airline']}<br>From {row['orig_code']} to {row['div_dest_code']}"
    ))
    # Original destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='star'),
        name=f'Original ({row["orig_code"]})',
        showlegend=(idx==0),
        text=[row['orig_code']],
        textposition='top center'
    ))
    # Diverted destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='diamond'),
        name=f'Diverted ({row["div_dest_code"]})',
        showlegend=(idx==0),
        text=[row['div_dest_code']],
        textposition='bottom center'
    ))

fig.update_layout(
    title='Flight Reroutes: Connected Original and Diverted Destinations',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()

In [ ]:
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Draw line connecting original and diverted destination
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon'], row['div_lon']],
        lat=[row['orig_lat'], row['div_lat']],
        mode='lines',
        line=dict(width=2, color=color),
        showlegend=False,
        hoverinfo='text',
        text=f"{row['airline']}<br>From {row['orig_code']} to {row['div_dest_code']}"
    ))
    # Original destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='star'),
        name=f'Original ({row["orig_code"]})',
        showlegend=(idx==0),
        text=[row['orig_code']],
        textposition='top center'
    ))
    # Diverted destination marker
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='diamond'),
        name=f'Diverted ({row["div_dest_code"]})',
        showlegend=(idx==0),
        text=[row['div_dest_code']],
        textposition='bottom center'
    ))

fig.update_layout(
    title='Flight Reroutes: Connected Original and Diverted Destinations',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()


In [ ]:
fig = go.Figure()

for idx, row in flight_rerouted.iterrows():
    airline = row['airline']
    color = airline_colors[airline]
    # Plot original and diverted destinations as stars (same color)
    fig.add_trace(go.Scattergeo(
        lon=[row['orig_lon']],
        lat=[row['orig_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='star'),
        name=f'Original Destination ({airline})',
        showlegend=(idx==0),
        text=[row['orig_code']],
        textposition='top center'
    ))
    fig.add_trace(go.Scattergeo(
        lon=[row['div_lon']],
        lat=[row['div_lat']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='star'),
        name=f'Diverted Destination ({airline})',
        showlegend=(idx==0),
        text=[row['div_dest_code']],
        textposition='bottom center'
    ))
    # Draw waypoint lines only if they differ
    if row['orig_waypoint'] != row['div_waypoint']:
        # Original waypoint connection (red)
        if row['orig_way_lat'] and row['orig_way_lon']:
            fig.add_trace(go.Scattergeo(
                lon=[row['orig_lon'], row['orig_way_lon']],
                lat=[row['orig_lat'], row['orig_way_lat']],
                mode='lines',
                line=dict(width=2, color='red'),
                showlegend=False,
                hoverinfo='text',
                text=f"Original Waypoint: {row['orig_waypoint']}"
            ))
        # Diverted waypoint connection (green)
        if row['div_way_lat'] and row['div_way_lon']:
            fig.add_trace(go.Scattergeo(
                lon=[row['div_lon'], row['div_way_lon']],
                lat=[row['div_lat'], row['div_way_lat']],
                mode='lines',
                line=dict(width=2, color='green'),
                showlegend=False,
                hoverinfo='text',
                text=f"Diverted Waypoint: {row['div_waypoint']}"
            ))

fig.update_layout(
    title='Flight Reroutes: Waypoint Differences Highlighted',
    geo=dict(scope='world'),
    width=2000,
    height=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)
fig.show()